# Notebook 4: Gold Layer - Business Aggregates

**Objective**: Create aggregated views ready for analysis and reporting.

## What is the Gold Layer?

The **Gold Layer** is the presentation layer of the lakehouse:
- **Aggregates**: Pre-computed data for performance
- **KPIs**: Key performance indicators
- **Optimized for reporting**: Fast queries for dashboards

## Data Flow

```
Silver (clean) ──► Aggregation ──► Gold (business ready)
```

---
## INITIALIZATION

In [ ]:
# Configure path to project
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import Spark modules and configuration
from lakehouse.spark_session import get_spark
from pyspark.sql.functions import sum as spark_sum, round as spark_round, count

# Create Spark session
spark = get_spark("gold-aggregations")

# Configure Nessie main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Create namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("=" * 60)
print("GOLD LAYER - BUSINESS AGGREGATES")
print("=" * 60)

## 1. Read Silver Table

The Gold layer reads from Silver (already cleaned data).

In [ ]:
# Read Silver table
sales_silver = spark.table("nessie.silver.sales")

silver_count = sales_silver.count()
print(f"Records in Silver: {silver_count:,}")

# Preview
print("\n=== Silver Preview ===")
sales_silver.select("order_id", "category", "region", "sales", "profit").show(5, truncate=False)

## 2. Create Main Aggregate: Sales by Category and Region

**KPIs calculated**:
- `total_sales`: Sum of sales
- `total_profit`: Sum of profits
- `total_quantity`: Total quantity sold
- `order_count`: Number of orders

This aggregate allows analyzing performance by Category × Region combination.

In [ ]:
# Aggregate by category and region
sales_by_category_region = (
    sales_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

print("=== Aggregate: Sales by Category × Region ===")
sales_by_category_region.orderBy("total_sales", ascending=False).show(truncate=False)

## 3. Other Useful Aggregates

In [ ]:
# Aggregate: Sales by Segment (Consumer, Corporate, Home Office)
from pyspark.sql.functions import avg as spark_avg

sales_by_segment = (
    sales_silver
    .groupBy("segment")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_round(spark_avg("sales"), 2).alias("avg_order_value"),
        count("order_id").alias("order_count")
    )
)

print("=== Aggregate: Sales by Segment ===")
sales_by_segment.orderBy("total_sales", ascending=False).show(truncate=False)

In [ ]:
# Aggregate: Top products
top_products = (
    sales_silver
    .groupBy("category", "sub_category", "product_name")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity")
    )
)

print("=== Top 10 Products ===")
top_products.orderBy("total_sales", ascending=False).show(10, truncate=False)

## 4. Drop Old Gold Tables (if they exist)

In [ ]:
# Cleanup: drop tables if they exist
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_category_region")
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_segment")
spark.sql("DROP TABLE IF EXISTS nessie.gold.top_products")

print("Existing tables dropped")

## 5. Create Gold Tables

In [ ]:
# Create Gold tables
sales_by_category_region.writeTo("nessie.gold.sales_by_category_region").using("iceberg").create()
sales_by_segment.writeTo("nessie.gold.sales_by_segment").using("iceberg").create()
top_products.writeTo("nessie.gold.top_products").using("iceberg").create()

print("Gold tables created:")
print("- nessie.gold.sales_by_category_region")
print("- nessie.gold.sales_by_segment")
print("- nessie.gold.top_products")

## 6. Verify Gold Tables

In [ ]:
# List all Gold tables
print("=== Tables in Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

In [ ]:
# Verify records
print("=== Record Counts ===")
spark.sql("SELECT 'sales_by_category_region' as table_name, COUNT(*) as count FROM nessie.gold.sales_by_category_region").show()
spark.sql("SELECT 'sales_by_segment' as table_name, COUNT(*) as count FROM nessie.gold.sales_by_segment").show()
spark.sql("SELECT 'top_products' as table_name, COUNT(*) as count FROM nessie.gold.top_products").show()

## 7. Analytical Query Examples

Gold tables are optimized for business queries.

In [ ]:
# Top 5 by sales
print("=== Top 5 Category × Region ===")
spark.sql("""
    SELECT category, region, total_sales, total_profit, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Best margin by region
print("=== Profit Margin by Region ===")
spark.sql("""
    SELECT 
        region,
        SUM(total_sales) as sales,
        SUM(total_profit) as profit,
        ROUND(100.0 * SUM(total_profit) / SUM(total_sales), 2) as margin_pct
    FROM nessie.gold.sales_by_category_region
    GROUP BY region
    ORDER BY profit DESC
""").show(truncate=False)

In [ ]:
# Performance by category
print("=== Performance by Category ===")
spark.sql("""
    SELECT 
        category,
        SUM(total_sales) as global_sales,
        SUM(total_profit) as global_profit,
        SUM(order_count) as global_orders
    FROM nessie.gold.sales_by_category_region
    GROUP BY category
    ORDER BY global_sales DESC
""").show(truncate=False)

In [ ]:
# Performance by segment
print("=== Performance by Segment ===")
spark.sql("""
    SELECT 
        segment,
        total_sales,
        total_profit,
        avg_order_value,
        order_count
    FROM nessie.gold.sales_by_segment
    ORDER BY total_sales DESC
""").show(truncate=False)

## COMPLETE LAKEHOUSE SUMMARY

In [ ]:
# Get counts from each layer
bronze_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.bronze.sales").first()[0]
silver_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.silver.sales").first()[0]
gold_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.gold.sales_by_category_region").first()[0]

print("""
╔══════════════════════════════════════════════════════════╗
║              COMPLETE LAKEHOUSE - SUMMARY                ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Medallion Architecture:                                 ║
║                                                          ║
║  ┌─────────┐     ┌─────────┐     ┌─────────────┐         ║
║  │ BRONZE  │────►│ SILVER  │────►│     GOLD    │         ║
║  │  Raw    │     │  Clean  │     │  Aggregates │         ║
║  └─────────┘     └─────────┘     └─────────────┘         ║
║    {:>8,}         {:>8,}             {:>8,}              ║
║                                                          ║
║  Tables created:                                         ║
║    Bronze:  nessie.bronze.sales                          ║
║    Silver:  nessie.silver.sales                          ║
║    Gold:    nessie.gold.sales_by_category_region         ║
║             nessie.gold.sales_by_segment                 ║
║             nessie.gold.top_products                     ║
║                                                          ║
║  Technologies:                                           ║
║    • Apache Spark (processing)                           ║
║    • Apache Iceberg (ACID table format)                  ║
║    • Project Nessie (versioned catalog)                  ║
║    • AWS S3 (storage)                                    ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""".format(bronze_count, silver_count, gold_count))

print("Bronze → Silver → Gold pipeline completed successfully!")
print("Next step: Run '06_complete_demo.ipynb' for Nessie demo")